# Session 3 — Pixel Losses and Honest Scores 🧪

**Time:** 35–40 minutes  
**Goal:** score a segmentation prediction while excluding ignored border pixels.

You will work with tiny masks where every count is visible, then use the same accumulation pattern used for a validation set.

## 0. Setup

This lab has no download and no training step. Run the cells in order.

In [1]:
import torch
import torch.nn.functional as F

torch.manual_seed(7)
IGNORE_INDEX = 255
CLASS_NAMES = ['background', 'pet']
print('PyTorch:', torch.__version__)

PyTorch: 2.11.0+cpu


## 1. From logits to a mask (5 min)

A model gives two **logits** per pixel: one for background and one for pet. Softmax turns each pair into probabilities that sum to one; `argmax` selects the larger probability.

In [2]:
# Shape: [batch, classes, height, width]
logits = torch.tensor([[[[3.0, -1.0], [0.2, 1.2]],
                        [[-1.0, 3.0], [1.5, 0.3]]]])
probabilities = logits.softmax(dim=1)
prediction = probabilities.argmax(dim=1)

print('logits shape:       ', tuple(logits.shape))
print('probability sums:\n', probabilities.sum(dim=1))
print('prediction shape:   ', tuple(prediction.shape))
print('predicted labels:\n', prediction[0])
assert torch.allclose(probabilities.sum(dim=1), torch.ones_like(probabilities[:, 0]))
assert prediction.shape == (1, 2, 2)

logits shape:        (1, 2, 2, 2)
probability sums:
 tensor([[[1., 1.],
         [1., 1.]]])
prediction shape:    (1, 2, 2)
predicted labels:
 tensor([[0, 1],
        [1, 0]])


## 2. Ignored borders do not vote (6 min)

Oxford-IIIT Pet trimaps use `255` for an ambiguous border. It is neither background nor pet, so it must be excluded from both the loss and every metric.

In [3]:
targets = torch.tensor([[[0, 1], [IGNORE_INDEX, 0]]])
valid = targets != IGNORE_INDEX
loss = F.cross_entropy(logits, targets, ignore_index=IGNORE_INDEX)

print('valid pixels:', int(valid.sum()), 'of', targets.numel())
print('valid mask:\n', valid[0])
print(f'cross-entropy on valid pixels only: {loss.item():.3f}')
assert int(valid.sum()) == 3

valid pixels: 3 of 4
valid mask:
 tensor([[ True,  True],
        [False,  True]])
cross-entropy on valid pixels only: 0.126


## 3. Exercise E1 — accumulate only valid pixels (10 min)

Complete the helper below. It must update a 2×2 confusion matrix whose **rows are true classes** and **columns are predicted classes**. Ignore every target equal to `255`.

In [4]:
# TODO: update the confusion matrix using only target != IGNORE_INDEX pixels
def update_confusion(confusion, target, predicted, ignore_index=IGNORE_INDEX, num_classes=2):
    # 1. Filtrar únicamente los píxeles válidos (excluir 255)
    valid = target != ignore_index
    t_valid = target[valid]
    p_valid = predicted[valid]

    # 2. Acumular las coincidencias en la matriz de confusión [clase_real, clase_predicha]
    for t in range(num_classes):
        for p in range(num_classes):
            confusion[t, p] += ((t_valid == t) & (p_valid == p)).sum()

    return confusion

def scores_from_confusion(confusion):
    # Pixel Accuracy: (Suma de la diagonal) / (Total de píxeles válidos)
    total_pixels = confusion.sum().float()
    correct_pixels = torch.diag(confusion).sum().float()
    accuracy = correct_pixels / total_pixels

    # IoU por clase: TP / (TP + FP + FN)
    num_classes = confusion.shape[0]
    class_iou = torch.zeros(num_classes)
    for c in range(num_classes):
        tp = confusion[c, c].float()
        fp = confusion[:, c].sum().float() - tp
        fn = confusion[c, :].sum().float() - tp
        denominator = tp + fp + fn
        class_iou[c] = tp / denominator if denominator > 0 else 0.0

    return accuracy, class_iou

In [5]:
# Two small saved predictions: accumulate counts first, then score once.
batch_targets = [
    torch.tensor([[0, 1], [1, IGNORE_INDEX]]),
    torch.tensor([[0, 0], [1, 1]]),
]
batch_predictions = [
    torch.tensor([[0, 1], [0, 1]]),
    torch.tensor([[0, 1], [1, 0]]),
]

confusion = torch.zeros((2, 2), dtype=torch.int64)
for target, predicted in zip(batch_targets, batch_predictions):
    confusion = update_confusion(confusion, target, predicted)

accuracy, class_iou = scores_from_confusion(confusion)
print('rows=true, columns=predicted')
print(confusion)
print(f'dataset pixel accuracy: {accuracy:.3f}')
print(f'background IoU: {class_iou[0]:.3f}; pet IoU: {class_iou[1]:.3f}')
assert confusion.sum().item() == 7  # one of eight cells was ignored
assert torch.all((class_iou >= 0) & (class_iou <= 1))

rows=true, columns=predicted
tensor([[2, 1],
        [2, 2]])
dataset pixel accuracy: 0.571
background IoU: 0.400; pet IoU: 0.400


**Why accumulate?** Averaging IoU separately for each image gives every image equal weight, even if one has many more valid pixels. A validation report should state its aggregation rule; here we total counts over the dataset before calculating the score.

### 1. `update_confusion`
Filtra los bordes ignorados (`target != 255`) y suma los aciertos y errores en la matriz de $2 \times 2$:
* **Filas**: Clase real (*Ground Truth*).
* **Columnas**: Clase predicha.
* **Filtro**: Ignora cualquier celda con valor 255 antes de contar.

### 2. `scores_from_confusion`
* **Pixel Accuracy**: Suma de la diagonal dividida entre el total de píxeles válidos.
* **IoU por Clase**: $\frac{TP}{TP + FP + FN}$
  * **$TP$**: Aciertos de la clase (diagonal).
  * **$FP$**: Falsos positivos (suma de la columna $- TP$).
  * **$FN$**: Falsos negativos (suma de la fila $- TP$).

### 3. ¿Por qué acumular la matriz?
Calcula el rendimiento a nivel **global del dataset**. Promediar el IoU foto por foto distorsiona la evaluación, ya que le daría el mismo peso a una imagen pequeña de 10 píxeles que a una grande de 10,000.

## 4. Exercise E2 — the worked 4×4 pet IoU (8 min)

For the pet class, find `TP`, `FP`, and `FN`. The ignored lower-right cell must disappear before counting. The expected result is $3/(3+1+1)=0.60$.

In [6]:
target_4x4 = torch.tensor([
    [1, 1, 0, 0],
    [1, 1, 0, 0],
    [0, 0, 0, 0],
    [0, 0, 0, IGNORE_INDEX],
])
predicted_4x4 = torch.tensor([
    [1, 1, 0, 0],
    [1, 0, 1, 0],
    [0, 0, 0, 0],
    [0, 0, 0, 1],
])

In [7]:
# TODO: calculate the pet TP, FP, FN, and IoU while excluding ignored labels
# 1. Crear máscara de píxeles válidos (filtrando IGNORE_INDEX)
valid = target_4x4 != IGNORE_INDEX

valid_targets = target_4x4[valid]
valid_preds = predicted_4x4[valid]

# 2. Calcular TP, FP y FN para la clase 'pet' (clase 1)
pet_tp = ((valid_targets == 1) & (valid_preds == 1)).sum().item()
pet_fp = ((valid_targets == 0) & (valid_preds == 1)).sum().item()
pet_fn = ((valid_targets == 1) & (valid_preds == 0)).sum().item()

# 3. Calcular el IoU de la clase 'pet'
pet_iou = pet_tp / (pet_tp + pet_fp + pet_fn)

print(f"TP: {pet_tp}, FP: {pet_fp}, FN: {pet_fn}")
print(f"Pet IoU: {pet_iou:.2f}")

assert pet_tp == 3
assert pet_fp == 1
assert pet_fn == 1
assert round(pet_iou, 2) == 0.60

TP: 3, FP: 1, FN: 1
Pet IoU: 0.60


## 5. Accuracy can hide a bad foreground mask (5 min)

Suppose a 10×10 image contains only four pet pixels. A model predicts every pixel as background. It is correct on 96 of 100 pixels, but it finds none of the pet.

In [8]:
rare_target = torch.zeros((10, 10), dtype=torch.int64)
rare_target[:2, :2] = 1
all_background = torch.zeros_like(rare_target)

rare_confusion = torch.zeros((2, 2), dtype=torch.int64)
rare_confusion = update_confusion(rare_confusion, rare_target, all_background)
rare_accuracy, rare_iou = scores_from_confusion(rare_confusion)
print(f'pixel accuracy: {rare_accuracy:.2%}')
print(f'pet IoU: {rare_iou[1]:.2f}')
assert rare_accuracy == 0.96 and rare_iou[1] == 0.0

pixel accuracy: 96.00%
pet IoU: 0.00


> TODO: In one or two sentences, explain why the 96% pixel accuracy above is
misleading.


Pixel accuracy is misleading because it is heavily dominated by the majority background class (which accounts for 96% of the image), hiding the fact that the model completely failed to detect the pet. In contrast, **IoU** exposes this failure by directly measuring the true overlap for the foreground class of interest.

## Checkpoint

Before leaving, confirm that you can explain: (1) why `255` is excluded, (2) why the 4×4 pet IoU is 0.60, and (3) why a dataset metric starts from accumulated counts. Next session you will use these metrics to evaluate a pretrained segmentation model.